In [6]:
import ratespricer as rp
import template as tp
from datetime import datetime
import pandas as pd

D:\2.app_install\Anaconda\lib\site-packages\mpl_finance.py:16: DeprecationWarning: 



    Please use `mplfinance` instead (no hyphen, no underscore).

    To install: `pip install --upgrade mplfinance` 

   For more information, see: https://pypi.org/project/mplfinance/


  __warnings.warn('\n\n  ================================================================='+


In [ ]:
fut_code="TL2309"
bond_code="220024"
start_date=datetime(2023, 7, 10)
end_date=datetime(2023, 7, 11)

In [17]:
def get_fut_rate(fut_code, bond_code, start_date, end_date):
    fut_code = fut_code.upper()
    fut = tp.futures_data_reader(fut_code, start_date, end_date).resample("60s").last().ffill()
    fut_am = fut.between_time("09:30", "11:30")
    fut_pm = fut.between_time("13:30", "15:15")
    fut = pd.concat([fut_am, fut_pm], axis=0)
    fut_rate = fut["fut_price"].apply(lambda x: rp.f_implied_yield(fut_code, x, bond_code))
    df = pd.DataFrame()
    df['close']=fut["fut_price"]
    df["fut_rate"] = fut_rate
    df["fut_code"]=str(fut_code)
    df["bond_code"]=str(bond_code)

    df = df.sort_index(ascending=False)
    df = df.dropna().reset_index()
    return df

In [ ]:
%%time
get_fut_rate(fut_code, bond_code, start_date, end_date)

In [ ]:
fut_code, bond_code, start_date, end_date

In [ ]:
df_data= pd.read_csv(r'C:\Users\Administrator.DESKTOP-S17J2N3\Desktop\ctd\国债期货可交割国债now.csv')
df_out= pd.DataFrame()
df_out['fut_code']=df_data['期货代码']
df_out['bond_code']=df_data['可交割国债'] 
df_out

In [ ]:
dff_all=pd.DataFrame(columns=['datetimes','close','fut_rate','fut_code','bond_code'])
start=datetime(2023, 7, 13)
end=datetime.today()
for fut,bond in zip(df_out['fut_code'],df_out['bond_code']):
    print(fut)
    print(bond)
    print(start)
    print(end)
    data=get_fut_rate(fut,str(bond),start,end)
    print(data)
#     dff_all.append(data)
    dff_all = pd.concat([dff_all, data], axis=0)
    print(dff_all)
dff_all.to_csv(r'C:\Users\Administrator.DESKTOP-S17J2N3\Desktop\ctd\ctdrate-now.csv')
    

In [ ]:
dff_all=pd.DataFrame(columns=['datetimes','close','fut_rate','fut_code','bond_code'])
end=datetime.today()
for fut,bond,start in zip(df_out['fut_code'],df_out['bond_code'],df_out['start']):
    print(fut)
    print(bond)
    start_1=datetime(*map(int,start.split('-')))
    print(start_1)
    print(end)
    data=get_fut_rate(fut,str(bond),start_1,end)
    print(data)
#     dff_all.append(data)
    dff_all = pd.concat([dff_all, data], axis=0)
    print(dff_all)
dff_all.to_csv(r'C:\Users\Administrator.DESKTOP-S17J2N3\Desktop\ctd\ctdrate-now.csv')
    

In [ ]:
data

In [19]:
import os
from datetime import datetime
import numpy as np
import pandas as pd
import pymysql

In [ ]:
dff_all

In [ ]:
cnn = pymysql.connect(host='192.168.119.55', user='intern', passwd='rates2022C!', db='bond', charset='utf8')
cur = cnn.cursor()
sql_upload = 'REPLACE INTO futures_rate_1m(DateTime,Close,Rate,FutCode,BondCode) ' \
      'values (%s,%s,%s,%s,%s)'
cur.executemany(sql_upload,dff_all.values.tolist())
cnn.commit()
print("上传完成！")
cur.close()


In [ ]:
from pandas import DataFrame,Series
def fut_ctdrate_reader(fut_code, ctd_code, start_date, end_date):
    cnn = pymysql.connect(host='192.168.119.55', user='intern', passwd='rates2022C!', db='bond', charset='utf8')
    cur = cnn.cursor()
    select_fut = '''
                 SELECT DateTime, FutCode,BondCode,Rate FROM bond.futures_rate_1m 
                 where FutCode = '%s' and BondCode = '%s' and DateTime BETWEEN '%s' AND '%s' ORDER BY DateTime
                  '''
    cur.execute(select_fut % (fut_code, ctd_code, start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d')))
    rst_fut = cur.fetchall()
    cur.close()
    cnn.close()
    if len(rst_fut) < 2:
        return []
    # 处理
    fut = DataFrame.from_records(list(rst_fut), columns=['datetimes','FutCode','BondCode', 'Rate']).set_index(['datetimes'])
    return fut

In [ ]:
fut_code="TL2309"
bond_code="220024"
start_date=datetime(2023, 7, 10)
end_date=datetime(2023, 7, 11)
a=fut_ctdrate_reader(fut_code, bond_code, start_date, end_date)
a['datetime']=a.index
a

In [1]:
from WindPy import w
w.start()

Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2020 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.


.ErrorCode=0
.Data=[OK!]

In [13]:
def get_ctdlist(futcode,end_date):
    ctdlist=pd.DataFrame()
    ctdlist['ctd']=w.wset("deliverablebondlist","windcode=" + futcode + ";date=" + str(end_date) + ";flag=interbank;pricetype=close").Data[0]
    ctdlist['futcode'] = futcode
    return ctdlist

In [ ]:
get_ctdlist("T2309.CFE",'2023-07-14')

In [ ]:

result = pd.DataFrame()
for futcode in fut_codes:
    futcode = futcode + '.CFE'
    dff = get_ctdlist(futcode,'2023-07-14')
    result = pd.concat([result, dff], axis=0)

In [3]:
futcode_list = result['futcode'].to_list()
ctd_list = result['ctd'].to_list()
start_1 = datetime.today()
end = datetime.today()

re_all = pd.DataFrame()
for fut,bond in zip(futcode_list,ctd_list):
    fut = fut[:-4]
    re_in = pd.DataFrame()
    bond = bond[:-3]
    ree = get_fut_rate(fut,bond,start_1,end)
    print(ree)
    re_in = pd.concat([re_in, ree], axis=0)
    re_all = pd.concat([re_all, re_in], axis=0)
re_all

NameError: name 'result' is not defined

In [ ]:
cnn = pymysql.connect(host='192.168.119.55', user='intern', passwd='rates2022C!', db='bond', charset='utf8')
cur = cnn.cursor()
sql_ctd = 'REPLACE INTO futures_rate_1m(DateTime,Close,Rate,FutCode,BondCode) ' \
      'values (%s,%s,%s,%s,%s)'
cur.executemany(sql_ctd,re_all.values.tolist())
cnn.commit()
print("ctd上传完成！")
cur.close()

In [15]:
def u_fut_rate_1m(fut_code,start_date,end_date):
    if not w.isconnected():
        w.start()
    futcfe = fut_code + '.CFE'
    result = pd.DataFrame()
    dff = get_ctdlist(futcfe,start_date)
    result = pd.concat([result, dff], axis=0)
    futcode_list = result['futcode'].to_list()
    ctd_list = result['ctd'].to_list()
    re_all = pd.DataFrame()
    for futcfe,bond in zip(futcode_list,ctd_list):
        fut = futcfe[:-4]
        re_in = pd.DataFrame()
        bond = bond[:-3]
        ree = get_fut_rate(fut,bond,start_date,end_date)
        print(ree)
        re_in = pd.concat([re_in, ree], axis=0)
        re_all = pd.concat([re_all, re_in], axis=0)
    print(re_all)
    cnn = pymysql.connect(host='192.168.119.55', user='intern', passwd='rates2022C!', db='bond', charset='utf8')
    cur = cnn.cursor()
    sql_ctd = 'REPLACE INTO futures_rate_1m(DateTime,Close,Rate,FutCode,BondCode) ' \
          'values (%s,%s,%s,%s,%s)'
    cur.executemany(sql_ctd,re_all.values.tolist())
    cnn.commit()
    print("%sctd上传完成！"%end_date)
    cur.close()

In [20]:
dm_1, dm_2, dm_3 =  '2309', '2312', '2403'  # delivery month
fut_codes = ['TL'  + dm_1, 'TL'  + dm_2, 'TL'  + dm_3,
                'T'  + dm_1, 'T'  + dm_2, 'T'  + dm_3,
                 'TF' + dm_1, 'TF' + dm_2, 'TF' + dm_3,
                 'TS' + dm_1, 'TS' + dm_2, 'TS' + dm_3]


date_input = datetime.today()
for fut in fut_codes:
    print(date_input)
    u_fut_rate_1m(fut, date_input, date_input)

2023-07-14 18:43:08.436165
              datetimes  close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  98.37    3.1003   TL2309    180024
1   2023-07-14 15:14:00  98.37    3.1003   TL2309    180024
2   2023-07-14 15:13:00  98.37    3.1003   TL2309    180024
3   2023-07-14 15:12:00  98.36    3.1009   TL2309    180024
4   2023-07-14 15:11:00  98.36    3.1009   TL2309    180024
..                  ...    ...       ...      ...       ...
222 2023-07-14 09:34:00  98.26    3.1071   TL2309    180024
223 2023-07-14 09:33:00  98.25    3.1077   TL2309    180024
224 2023-07-14 09:32:00  98.26    3.1071   TL2309    180024
225 2023-07-14 09:31:00  98.25    3.1077   TL2309    180024
226 2023-07-14 09:30:00  98.30    3.1046   TL2309    180024

[227 rows x 5 columns]
              datetimes  close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  98.37    3.0972   TL2309    190010
1   2023-07-14 15:14:00  98.37    3.0972   TL2309    190010
2   2023-07-14 15:13:00  98.37    3.0972   TL2309

              datetimes  close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  97.93    3.1156   TL2312    210014
1   2023-07-14 15:14:00  97.93    3.1156   TL2312    210014
2   2023-07-14 15:13:00  97.95    3.1145   TL2312    210014
3   2023-07-14 15:12:00  97.95    3.1145   TL2312    210014
4   2023-07-14 15:11:00  97.95    3.1145   TL2312    210014
..                  ...    ...       ...      ...       ...
221 2023-07-14 09:35:00  97.83    3.1213   TL2312    210014
222 2023-07-14 09:34:00  97.84    3.1207   TL2312    210014
223 2023-07-14 09:33:00  97.83    3.1213   TL2312    210014
224 2023-07-14 09:32:00  97.83    3.1213   TL2312    210014
225 2023-07-14 09:31:00  97.84    3.1207   TL2312    210014

[226 rows x 5 columns]
              datetimes  close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  97.93    3.1188   TL2312    200004
1   2023-07-14 15:14:00  97.93    3.1188   TL2312    200004
2   2023-07-14 15:13:00  97.95    3.1176   TL2312    200004
3   2023-07-14 1

              datetimes  close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  97.55    3.1314   TL2403    220024
1   2023-07-14 15:14:00  97.55    3.1314   TL2403    220024
2   2023-07-14 15:13:00  97.55    3.1314   TL2403    220024
3   2023-07-14 15:12:00  97.55    3.1314   TL2403    220024
4   2023-07-14 15:11:00  97.55    3.1314   TL2403    220024
..                  ...    ...       ...      ...       ...
221 2023-07-14 09:35:00  97.47    3.1357   TL2403    220024
222 2023-07-14 09:34:00  97.47    3.1357   TL2403    220024
223 2023-07-14 09:33:00  97.47    3.1357   TL2403    220024
224 2023-07-14 09:32:00  97.44    3.1374   TL2403    220024
225 2023-07-14 09:31:00  97.44    3.1374   TL2403    220024

[226 rows x 5 columns]
              datetimes  close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  97.55    3.1487   TL2403    190010
1   2023-07-14 15:14:00  97.55    3.1487   TL2403    190010
2   2023-07-14 15:13:00  97.55    3.1487   TL2403    190010
3   2023-07-14 1

              datetimes    close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  101.965    2.7393    T2309    220003
1   2023-07-14 15:14:00  101.965    2.7393    T2309    220003
2   2023-07-14 15:13:00  101.980    2.7374    T2309    220003
3   2023-07-14 15:12:00  101.970    2.7387    T2309    220003
4   2023-07-14 15:11:00  101.960    2.7400    T2309    220003
..                  ...      ...       ...      ...       ...
222 2023-07-14 09:34:00  101.905    2.7472    T2309    220003
223 2023-07-14 09:33:00  101.910    2.7465    T2309    220003
224 2023-07-14 09:32:00  101.910    2.7465    T2309    220003
225 2023-07-14 09:31:00  101.925    2.7446    T2309    220003
226 2023-07-14 09:30:00  101.925    2.7446    T2309    220003

[227 rows x 5 columns]
              datetimes    close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  101.965    2.6785    T2309    200006
1   2023-07-14 15:14:00  101.965    2.6785    T2309    200006
2   2023-07-14 15:13:00  101.980    2.6760    

              datetimes    close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  101.540    2.8054    T2312    220025
1   2023-07-14 15:14:00  101.540    2.8054    T2312    220025
2   2023-07-14 15:13:00  101.565    2.8023    T2312    220025
3   2023-07-14 15:12:00  101.555    2.8036    T2312    220025
4   2023-07-14 15:11:00  101.560    2.8029    T2312    220025
..                  ...      ...       ...      ...       ...
222 2023-07-14 09:34:00  101.515    2.8086    T2312    220025
223 2023-07-14 09:33:00  101.515    2.8086    T2312    220025
224 2023-07-14 09:32:00  101.520    2.8080    T2312    220025
225 2023-07-14 09:31:00  101.530    2.8067    T2312    220025
226 2023-07-14 09:30:00  101.545    2.8048    T2312    220025

[227 rows x 5 columns]
              datetimes    close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  101.540    2.7943    T2312    220010
1   2023-07-14 15:14:00  101.540    2.7943    T2312    220010
2   2023-07-14 15:13:00  101.565    2.7910    

              datetimes    close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  101.180    2.8495    T2403    230004
1   2023-07-14 15:14:00  101.180    2.8495    T2403    230004
2   2023-07-14 15:13:00  101.185    2.8488    T2403    230004
3   2023-07-14 15:12:00  101.185    2.8488    T2403    230004
4   2023-07-14 15:11:00  101.185    2.8488    T2403    230004
..                  ...      ...       ...      ...       ...
222 2023-07-14 09:34:00  101.155    2.8526    T2403    230004
223 2023-07-14 09:33:00  101.145    2.8539    T2403    230004
224 2023-07-14 09:32:00  101.165    2.8513    T2403    230004
225 2023-07-14 09:31:00  101.170    2.8507    T2403    230004
226 2023-07-14 09:30:00  101.190    2.8482    T2403    230004

[227 rows x 5 columns]
              datetimes    close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  101.180    2.8466    T2403    220025
1   2023-07-14 15:14:00  101.180    2.8466    T2403    220025
2   2023-07-14 15:13:00  101.185    2.8459    

              datetimes    close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  102.095    2.5128   TF2309    230008
1   2023-07-14 15:14:00  102.095    2.5128   TF2309    230008
2   2023-07-14 15:13:00  102.105    2.5105   TF2309    230008
3   2023-07-14 15:12:00  102.100    2.5116   TF2309    230008
4   2023-07-14 15:11:00  102.100    2.5116   TF2309    230008
..                  ...      ...       ...      ...       ...
222 2023-07-14 09:34:00  102.030    2.5277   TF2309    230008
223 2023-07-14 09:33:00  102.035    2.5265   TF2309    230008
224 2023-07-14 09:32:00  102.035    2.5265   TF2309    230008
225 2023-07-14 09:31:00  102.050    2.5231   TF2309    230008
226 2023-07-14 09:30:00  102.060    2.5208   TF2309    230008

[227 rows x 5 columns]
              datetimes    close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  102.095    2.4601   TF2309    220022
1   2023-07-14 15:14:00  102.095    2.4601   TF2309    220022
2   2023-07-14 15:13:00  102.105    2.4575   T

              datetimes    close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  101.320    2.3489   TS2309    200013
1   2023-07-14 15:14:00  101.320    2.3489   TS2309    200013
2   2023-07-14 15:13:00  101.330    2.3441   TS2309    200013
3   2023-07-14 15:12:00  101.330    2.3441   TS2309    200013
4   2023-07-14 15:11:00  101.330    2.3441   TS2309    200013
..                  ...      ...       ...      ...       ...
222 2023-07-14 09:34:00  101.270    2.3733   TS2309    200013
223 2023-07-14 09:33:00  101.275    2.3708   TS2309    200013
224 2023-07-14 09:32:00  101.270    2.3733   TS2309    200013
225 2023-07-14 09:31:00  101.270    2.3733   TS2309    200013
226 2023-07-14 09:30:00  101.275    2.3708   TS2309    200013

[227 rows x 5 columns]
              datetimes    close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  101.320    2.1017   TS2309    230005
1   2023-07-14 15:14:00  101.320    2.1017   TS2309    230005
2   2023-07-14 15:13:00  101.330    2.0950   T

              datetimes    close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  101.195    2.4285   TS2312    230003
1   2023-07-14 15:14:00  101.195    2.4285   TS2312    230003
2   2023-07-14 15:13:00  101.205    2.4238   TS2312    230003
3   2023-07-14 15:12:00  101.205    2.4238   TS2312    230003
4   2023-07-14 15:11:00  101.210    2.4215   TS2312    230003
..                  ...      ...       ...      ...       ...
222 2023-07-14 09:34:00  101.150    2.4498   TS2312    230003
223 2023-07-14 09:33:00  101.150    2.4498   TS2312    230003
224 2023-07-14 09:32:00  101.155    2.4474   TS2312    230003
225 2023-07-14 09:31:00  101.155    2.4474   TS2312    230003
226 2023-07-14 09:30:00  101.165    2.4427   TS2312    230003

[227 rows x 5 columns]
              datetimes    close  fut_rate fut_code bond_code
0   2023-07-14 15:15:00  101.195    2.2623   TS2312    220018
1   2023-07-14 15:14:00  101.195    2.2623   TS2312    220018
2   2023-07-14 15:13:00  101.205    2.2563   T